# 📝 深度学习 Padding (填充) 机制专题笔记

## 一、 CNN 中的 Padding (视觉特征图层级)

### 1. 核心概念与作用
* **本质**：在原始图像或特征图的边缘虚空捏造一圈“0”（或其他特定值）。
* **核心目的**：
    1.  **控制输出尺寸**：防止每次卷积后图像越来越小。
    2.  **保护边缘信息**：让图像边缘的像素也能像中心像素一样，多次被卷积核的中心扫过。

### 2. 尺寸计算与下采样对齐
计算单层输出特征图尺寸的核心公式（向下取整）：
$$O = \lfloor \frac{I - K + 2P}{S} \rfloor + 1$$
*(O: 输出尺寸, I: 输入尺寸, K: 卷积核大小, P: Padding, S: 步长 Stride)*

* **经典下采样配置 (完美减半)**：当输入 $16 \times 16$，想要输出 $8 \times 8$ 时，若使用 $3 \times 3$ 的卷积核 ($K=3$)，必须配置步长 $S=2$ 以及 **填充 $P=1$**。
* **YOLO 工程细节**：主流目标检测代码（如 Ultralytics）通常内置 `autopad` 函数，自动计算 $P = K // 2$ 以确保尺寸完美对齐。

### 3. Padding 的一个易错点
* **误区**：加了 Padding 或使用了下采样，会改变单个输出像素的感受野大小。
* **真相**：**单层卷积的感受野永远只和卷积核 $K$ 绑定。** 步长 $S$ 决定“看多少次”（缩小尺寸），Padding $P$ 决定“能不能看到边”，但每一次看进去的区域（感受野）依然是 $K \times K$（例如 $3 \times 3$）。

---

## 二、 Transformer 中的 Padding (序列层级)

### 1. 核心概念
* **本质**：为了让同一个 Batch 里的多条长短不一的句子能够组成规则的矩阵进行并行计算，强行用无意义的标记（如 `[PAD]`，对应 ID 通常为 0）把短句子补齐到统一的最大长度。
* **安全护城河：Attention Mask (Padding Mask)**
    * 在计算 Self-Attention 时，必须配合使用 Mask 矩阵，把这些无意义的 Padding 词的注意力得分强行设为负无穷大（如 $-10000$）。
    * 这样经过 Softmax 计算后，Padding 部分的概率就会变成 0，彻底被模型无视。

### 2. 核心灾难：推理 Padding 超越训练极限 (Length Extrapolation)
**高频提问**：训练时 Padding 到 512（Max Length），推理时 Padding 扩大到 1024 会发生什么？

#### 情况 A：真实文本很短（例如 300），只是为了凑整强行 Padding 到 1024
* **结果**：只要代码正确使用了 Attention Mask 将后面的 724 个 Padding 遮蔽，**模型准确率不受到任何影响**。
* **代价**：极度浪费显存和计算算力（多做了巨量乘以 0 的无用功）。

#### 情况 B：真实文本超过了 512（例如 800），剩余 Padding 到 1024
* **结果**：**灾难性故障。** 根据位置编码（PE）的设计不同，有两种死法：

    * **死法 1：直接报错崩溃 (Index Out of Bounds)**
        * **适用模型**：BERT、GPT-2 等使用**可学习的绝对位置编码**的模型。
        * **原因**：底层 PE 参数矩阵只有 512 行。物理层面上根本查不到索引为 513 的向量位置，代码当场崩溃。
    * **死法 2：输出胡言乱语/幻觉 (Softmax 分布失控)**
        * **适用模型**：原生 Transformer (正弦余弦 PE)、大语言模型 (RoPE 旋转位置编码)。
        * **原因 1（注意力稀释）**：即使 PE 能通过公式算出来，Softmax 的分母突然新增了 512 项。即使这些项原始得分不高，取 $e^x$ 后累加依然庞大，会将原本正确的注意力（如 80%）强行摊薄（如稀释到 40%）。
        * **原因 2（未知的距离共振/最致命）**：超纲的距离差（>512）产生了模型从未见过的三角函数/旋转角度叠加态。模型权重（$W_q, W_k$）未曾学习过如何压制这种未知组合，导致 $Q_i$ 和 $K_j$ 在高维空间方向意外重合。
        * **连锁反应**：点积得分（Logits）异常飙升（例如算得 15）。经 Softmax 指数放大（$e^{15}$）后触发“赢者通吃”，这一个异常项会抢走 99% 的注意力，彻底摧毁模型对真实上下文的理解。
    * **注**：公式中的缩放因子 $\sqrt{d}$ 只能压制“维度”带来的累加过大，对“长度”延伸产生的位置编码共振无能为力。